## **Step 1: Define the tools**

In [2]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
# This is raw model call. What we see in chat.openai.com has agent layer baked in as well.
llm = ChatOpenAI(model="gpt-5-mini") 

In [3]:
from langchain.tools import tool

In [4]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool to gather information about current events or general knowledge"""

    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    
    response = search.invoke(query)
    return response

    
tool_duckduckgo_search.invoke("What is the capital of France?")


'5 days ago -As the capital of France,Parisis the seat of France\'s national government. For the executive, the two chief officers each have their own official residences, which also serve as their offices. The President of the French Republic resides at the Élysée Palace. 1 day ago -Its capital, largest city and main cultural and economic centre is Paris.Metropolitan Francewas settled during the Iron Age by Celtic tribes known as Gauls before Rome annexed the area in 51 BC, leading to a distinct Gallo-Roman culture. March 10, 2026 -This is a chronological list of capitals of France. The capital of France has beenParissince its liberation in 1944. 2 weeks ago -Paris(nicknamed the "City of light") is the capital city of France, and the largest city in France. The area is 105 square kilometres (41 square miles), and around 2.15 million people live there. 6 days ago -France, a country of northwestern Europe, is historically and culturally among the most important countries in the Western 

In [5]:
@tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool to gather information about historical events"""

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)
    return response
    
tool_wikipedia_search.invoke("Alan Turing")


'Page: Alan Turing\nSummary: Alan Mathison Turing (; 23 June 1912 – 7 June 1954) was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. He was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorithm and computation with the Turing machine, which can be considered a model of a general-purpose computer. Turing is widely considered to be the father of theoretical computer science.\nBorn in London, Turing was raised in southern England. He graduated from King\'s College, Cambridge, and in 1938, earned a doctorate degree from Princeton University. During World War II, Turing worked for the Government Code and Cypher School at Bletchley Park, Britain\'s codebreaking centre that produced Ultra intelligence. He led Hut 8, the section responsible for German naval cryptanalysis. Turing devised techniques for speeding the breaking of German ciphers, including improvement

In [6]:
@tool
def tool_arxiv_search(query: str) -> str:
    """Use this tool to gather information about arXiv papers"""

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    #1. Initialize the ArxivAPIWrapper
    arxiv_api_wrapper = ArxivAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=4000
    )
    
    #2. Initialize the Arxiv Query Object
    arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)

    #3. Invoke the Arxiv Query Object
    response = arxiv.invoke(query)
    return response

tool_arxiv_search.invoke("What are the latest papers on AI?")

'Published: 2017-10-24\nTitle: Multi-messenger Observations of a Binary Neutron Star Merger\nAuthors: LIGO Scientific Collaboration, Virgo Collaboration, Fermi GBM, INTEGRAL, IceCube Collaboration, AstroSat Cadmium Zinc Telluride Imager Team, IPN Collaboration, The Insight-Hxmt Collaboration, ANTARES Collaboration, The Swift Collaboration, AGILE Team, The 1M2H Team, The Dark Energy Camera GW-EM Collaboration, the DES Collaboration, The DLT40 Collaboration, GRAWITA, :, GRAvitational Wave Inaf TeAm, The Fermi Large Area Telescope Collaboration, ATCA, :, Australia Telescope Compact Array, ASKAP, :, Australian SKA Pathfinder, Las Cumbres Observatory Group, OzGrav, DWF, AST3, CAASTRO Collaborations, The VINROUGE Collaboration, MASTER Collaboration, J-GEM, GROWTH, JAGWAR, Caltech- NRAO, TTU-NRAO, NuSTAR Collaborations, Pan-STARRS, The MAXI Team, TZAC Consortium, KU Collaboration, Nordic Optical Telescope, ePESSTO, GROND, Texas Tech University, SALT Group, TOROS, :, Transient Robotic Observat

In [13]:
@tool
def personal_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information. Expects a name as input."""
    
    infos = [
        {
            "name": "Ojas Dighe",
            "age": 25,
            "location": "India",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "John Doe",
            "age": 30,
            "location": "USA",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "Jane Smith",
            "age": 28,
            "location": "Canada",
            "interests": "coding, building things, reading, writing"
        }
    ]

    for info in infos:
        if info["name"] == query:
            return f"{info['name']} is {info['age']} years old and lives in {info['location']}. {info['name']} likes {info['interests']}"
    return "No information found"

personal_info.invoke("John Doe")

'John Doe is 30 years old and lives in USA. John Doe likes coding, building things, reading, writing'

In [9]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool to answer questions based on NovaSphere organization data"""

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings

    embed_model = OpenAIEmbeddings(model = 'text-embedding-3-small')

    chroma_db_conn = Chroma(
        embedding_function = embed_model,
        persist_directory = '../CH-2_RAG/vector_db'
    )
    #1. Retrieve the most relevant chunks from the vector database
    relevant_docs = chroma_db_conn.similarity_search(query, k=3)

    #2. Create a string of the most relevant chunks
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])

    return relevant_docs_content
    
tool_rag.invoke("What is NovaSphere?")

'Today, NovaSphere Technologies is considered a reliable organization that provides data\nToday, NovaSphere Technologies is considered a reliable organization that provides data\nToday, NovaSphere Technologies is considered a reliable organization that provides data'

## Bind Tools

In [ ]:
toolkit = [
            tool_duckduckgo_search, 
            tool_wikipedia_search, 
            tool_arxiv_search, 
            personal_info,
            tool_rag
          ]

# Step 1: Tool Binding
llm_bind = llm.bind_tools(toolkit)

llm_bind.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 230, 'total_tokens': 246, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUBXkDOkk1WwucFv4cKzhSAlgMzCq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d8702-899f-7113-a370-ae97aa70fc41-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 230, 'output_tokens': 16, 'total_tokens': 246, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
# We will not be able to get content in LLM response even after tool binding. 
# You will see tool calls in the response, but no content in AIMessage Object.
# To solve this, we need to use a tool calling agent.
llm_bind.invoke("Tell me about Ojas Dighe. Make tool calls if necessary")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 210, 'prompt_tokens': 236, 'total_tokens': 446, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUBXmtrxttmUraSVFWVgJudm0mQhZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8702-91db-7b13-b512-79baddfd4cbf-0', tool_calls=[{'name': 'personal_info', 'args': {'query': 'Ojas Dighe'}, 'id': 'call_gbj4lHtW90bRQeB9sZgq9Uee', 'type': 'tool_call'}, {'name': 'tool_duckduckgo_search', 'args': {'query': 'Ojas Dighe'}, 'id': 'call_QM46gfsPRUD0fIf0A4ZKIHHt', 'type': 'tool_call'}, {'name': 'tool_wikipedia_search', 'args': {'query': 'Ojas Dighe'}, 'id': 'call_YgMzubZWmNanxmQF

## **ReAct Agent**

In [16]:
from langchain.agents import create_agent

agent = create_agent(llm_bind,toolkit)

In [17]:
# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is the age of John Doe? Make tool calls if necessary"}]}
)

{'messages': [HumanMessage(content='What is the age of John Doe? Make tool calls if necessary', additional_kwargs={}, response_metadata={}, id='36974362-03a2-446d-89c8-2cbb26a41339'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 236, 'total_tokens': 260, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUBXpxDjxrBALdMAyqRmhCxVCq1H9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8702-a062-7050-bd3b-3f0c5e7625ec-0', tool_calls=[{'name': 'personal_info', 'args': {'query': 'John Doe'}, 'id': 'call_qRqK68SaYztdE20F8R1vuZq9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'in